# Taller: Políticas de Gobernanza para Arquitectura IA-Ready (Privacidad y Sesgo)

Objetivo: diseñar políticas de gobierno de datos para una plataforma IA/Big Data que asegure privacidad, cumplimiento y control de sesgos/deriva de modelos. Evaluaremos el cumplimiento de productos de datos y modelos con una matriz de políticas.

## 1. Ámbitos de política
- Privacidad y datos personales (PII/PHI/PCI)
- Acceso y control (RBAC/ABAC, mínimo privilegio)
- Ciclo de vida (retención/expurgo)
- Calidad/Observabilidad (SLOs, freshness, errores)
- Linaje y trazabilidad (auditoría)
- Responsable de modelo (MLOps): fairness, bias, drift, explainability

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
import pandas as pd

@dataclass
class DataAsset:
    key: str
    contains_pii: bool
    retention_days: int
    access_model: str  # 'RBAC' or 'ABAC'
    lineage_complete: bool
    freshness_slo_m: int
    observed: bool  # tiene métricas/alertas

@dataclass
class ModelAsset:
    key: str
    uses_protected_attr: bool
    fairness_metric: float  # e.g., disparate impact ratio
    fairness_min: float
    drift_monitoring: bool
    explainability: bool

policies = {
  'privacy': { 'pii_protection_required': True },
  'access': { 'allowed_models': ['RBAC','ABAC'] },
  'retention': { 'max_days': 365 },
  'observability': { 'freshness_max_m': 60, 'observed_required': True },
  'lineage': { 'complete_required': True },
  'fairness': { 'min_ratio': 0.8, 'no_protected_attr': True },
  'drift': { 'monitoring_required': True },
  'explain': { 'explainability_required': True }
}

assets = [
  DataAsset('silver.orders_clean', contains_pii=True, retention_days=180, access_model='RBAC', lineage_complete=True, freshness_slo_m=30, observed=True),
  DataAsset('gold.churn_scoring', contains_pii=True, retention_days=400, access_model='ABAC', lineage_complete=False, freshness_slo_m=90, observed=False)
]

models = [
  ModelAsset('models.churn_v1', uses_protected_attr=False, fairness_metric=0.86, fairness_min=0.8, drift_monitoring=True, explainability=True),
  ModelAsset('models.credit_risk_v2', uses_protected_attr=True, fairness_metric=0.75, fairness_min=0.8, drift_monitoring=False, explainability=False)
]

# Evaluación
issues: List[Dict] = []
for a in assets:
    if policies['privacy']['pii_protection_required'] and not a.contains_pii:
        pass  # ok si no contiene PII; si contiene, validación de controles se asume aparte
    if a.retention_days > policies['retention']['max_days']:
        issues.append({'asset': a.key, 'policy': 'retention', 'issue': f'retention_days={a.retention_days} > {policies[
][
]}'})
    if a.access_model not in policies['access']['allowed_models']:
        issues.append({'asset': a.key, 'policy': 'access', 'issue': f'access_model={a.access_model}'})
    if policies['lineage']['complete_required'] and not a.lineage_complete:
        issues.append({'asset': a.key, 'policy': 'lineage', 'issue': 'lineage incomplete'})
    if a.freshness_slo_m > policies['observability']['freshness_max_m']:
        issues.append({'asset': a.key, 'policy': 'observability', 'issue': f'freshness_slo_m={a.freshness_slo_m}'})
    if policies['observability']['observed_required'] and not a.observed:
        issues.append({'asset': a.key, 'policy': 'observability', 'issue': 'not observed'})

for m in models:
    if policies['fairness']['no_protected_attr'] and m.uses_protected_attr:
        issues.append({'asset': m.key, 'policy': 'fairness', 'issue': 'uses protected attribute'})
    if m.fairness_metric < policies['fairness']['min_ratio']:
        issues.append({'asset': m.key, 'policy': 'fairness', 'issue': f'fairness={m.fairness_metric} < {policies[
][
]}'})
    if policies['drift']['monitoring_required'] and not m.drift_monitoring:
        issues.append({'asset': m.key, 'policy': 'drift', 'issue': 'no drift monitoring'})
    if policies['explain']['explainability_required'] and not m.explainability:
        issues.append({'asset': m.key, 'policy': 'explain', 'issue': 'no explainability'})

pd.DataFrame(issues)

## 2. Plan de remediación y priorización
Ejemplos de acciones: reducir retención, completar lineage, instrumentar observabilidad, eliminar atributos protegidos, habilitar monitoreo de drift y explicabilidad.

Matriz de priorización sugerida: impacto en riesgo x esfuerzo.

## 3. Checklist de políticas IA-ready
- [ ] Clasificación de datos y PII identificada
- [ ] Retención conforme a normativa
- [ ] Acceso con RBAC/ABAC + mínimo privilegio
- [ ] Lineage visible y completo
- [ ] SLOs de freshness/errores con alertas
- [ ] Modelos sin atributos protegidos o con justificación y controles
- [ ] Fairness metrics ≥ umbral
- [ ] Monitoreo de drift
- [ ] Explicabilidad (post-hoc o intrínseca)
- [ ] Auditoría y DSAR (solicitudes de datos)